# Week 05 - Day 02 | Instance & Class Variables

**Topics:** instance variables · class variables · `@classmethod` · `@staticmethod` · factory methods · mutable default pitfall  
**Week:** OOP

---

## 1. Instance Variables vs Class Variables

| | Instance Variable | Class Variable |
|---|---|---|
| **Where defined** | Inside `__init__` via `self.x` | In the class body, outside methods |
| **Belongs to** | One specific object | The class itself — all instances share it |
| **Use for** | Data that differs per object | Data that is the same for every object |

In [ ]:
class Dog:
    species = 'Canis lupus familiaris'  # CLASS variable — shared by all

    def __init__(self, name, age):
        self.name = name  # INSTANCE variable — each dog has its own
        self.age  = age

rex  = Dog('Rex',  3)
luna = Dog('Luna', 5)

print(rex.name)     # Rex
print(luna.name)    # Luna
print(rex.species)  # Canis lupus familiaris
print(Dog.species)  # same — accessed directly on the class

## 2. Class Variables — Shared State

A classic use-case: counting how many instances have been created.

In [ ]:
class Employee:
    employee_count = 0  # class variable

    def __init__(self, name, department):
        self.name       = name
        self.department = department
        Employee.employee_count += 1  # update the CLASS variable

e1 = Employee('Alice', 'Engineering')
e2 = Employee('Bob',   'Marketing')
e3 = Employee('Carol', 'Engineering')

print(Employee.employee_count)  # 3
print(e1.employee_count)        # 3 — visible via instance too

## 3. The Shadowing Pitfall

Assigning `self.x = ...` when `x` is a class variable does **not** change the class variable.  
Python creates a new **instance** variable that *shadows* the class variable.

In [ ]:
class Config:
    debug = False

c1 = Config()
c2 = Config()

c1.debug = True      # creates an INSTANCE variable on c1 only

print(c1.debug)      # True  ← c1's own instance variable
print(c2.debug)      # False ← still reads the class variable
print(Config.debug)  # False ← class variable unchanged

# To change it for ALL instances, assign on the class:
Config.debug = True
print(c2.debug)      # True

## 4. Class Methods — `@classmethod`

A class method receives the **class** (not an instance) as its first argument — conventionally named `cls`.

Most common uses:
- **Alternative constructors** (factory methods)
- **Operating on class variables**

```python
@classmethod
def method_name(cls, ...):
    ...
```

In [ ]:
class Temperature:
    unit = 'Celsius'

    def __init__(self, celsius):
        self.celsius = celsius

    def __str__(self):
        return f'{self.celsius}°C'

    @classmethod
    def from_fahrenheit(cls, fahrenheit):
        return cls((fahrenheit - 32) * 5 / 9)

    @classmethod
    def from_kelvin(cls, kelvin):
        return cls(kelvin - 273.15)

t1 = Temperature(100)
t2 = Temperature.from_fahrenheit(212)
t3 = Temperature.from_kelvin(373.15)

print(t1)  # 100°C
print(t2)  # 100.0°C
print(t3)  # 100.0°C

## 5. Static Methods — `@staticmethod`

A static method receives **neither** `self` nor `cls`.  
It's a plain function that lives inside a class for organisational reasons.

Use it when the logic belongs to the class conceptually but doesn't need any instance or class data.

In [ ]:
class MathUtils:
    @staticmethod
    def is_even(n):
        return n % 2 == 0

    @staticmethod
    def clamp(value, minimum, maximum):
        return max(minimum, min(maximum, value))

    @staticmethod
    def percentage(part, total):
        if total == 0:
            return 0.0
        return round(part / total * 100, 2)

print(MathUtils.is_even(4))          # True
print(MathUtils.clamp(15, 0, 10))    # 10
print(MathUtils.percentage(3, 4))    # 75.0

## 6. Comparing All Three Method Types

| Type | First param | Receives | Typical use |
|------|-------------|----------|-------------|
| Instance method | `self` | the instance | read/modify instance data |
| Class method | `cls` | the class | factory methods, class vars |
| Static method | — | nothing | utility helpers |

In [ ]:
class Date:
    default_separator = '-'

    def __init__(self, year, month, day):
        self.year  = year
        self.month = month
        self.day   = day

    def format(self):  # instance method
        sep = Date.default_separator
        return f'{self.year}{sep}{self.month:02d}{sep}{self.day:02d}'

    @classmethod
    def from_string(cls, date_str):  # factory method
        year, month, day = map(int, date_str.split('-'))
        return cls(year, month, day)

    @classmethod
    def set_separator(cls, sep):  # modifies class var
        cls.default_separator = sep

    @staticmethod
    def is_leap_year(year):  # pure logic
        return (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)

d1 = Date(2025, 6, 15)
d2 = Date.from_string('2024-02-29')

print(d1.format())              # 2025-06-15
Date.set_separator('/')
print(d1.format())              # 2025/06/15
print(Date.is_leap_year(2024))  # True
print(Date.is_leap_year(2025))  # False

## 7. Mutable Class Variable — Classic Bug

> **Never** use a mutable object (list, dict) as a class variable if you want per-instance data.  
> All instances share the **same** list!

In [ ]:
# BUG — shared list
class BuggyStudent:
    grades = []  # shared by ALL instances!
    def __init__(self, name):
        self.name = name
    def add_grade(self, g):
        self.grades.append(g)

s1 = BuggyStudent('Alice')
s2 = BuggyStudent('Bob')
s1.add_grade(90)
print(s2.grades)  # [90] ← BUG! Bob sees Alice's grade

# FIX — initialise in __init__
class CorrectStudent:
    def __init__(self, name):
        self.name   = name
        self.grades = []  # each instance gets its own list
    def add_grade(self, g):
        self.grades.append(g)

s3 = CorrectStudent('Alice')
s4 = CorrectStudent('Bob')
s3.add_grade(90)
print(s4.grades)  # [] ← correct

## Summary

| Concept | Key Point |
|---------|----------|
| Instance variable | Defined in `__init__` via `self.x` — per-object |
| Class variable | Defined in class body — shared by ALL instances |
| `@classmethod` | Receives `cls`; used for factories & class var operations |
| `@staticmethod` | No `self`/`cls`; pure utility function inside class |
| Shadowing | `self.x = ...` on a class var creates an instance copy |
| Mutable default | Never use lists/dicts as class vars for per-object data |

> **Next:** Day 03 — Inheritance (`super()`, method override, `isinstance()`, `issubclass()`)